In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from typing import List
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
class custom_transformations:

    def dedup(self,df:DataFrame,target_cols:List,latest_time):
        df = df.withColumn('dedupconcat',concat(*target_cols))
        df = df.withColumn('dedupCount',row_number().over(Window.partitionBy('dedupconcat').orderBy(desc(latest_time))))
        df = df.filter(col('dedupCount')==1)
        df = df.drop('dedupconcat','dedupCount')
        return df

    
    def proceess_date(self,df:DataFrame):
        df = df.withColumn('process_timestamp',current_timestamp())
        return df

    def upsert(self, df, key_cols, table, cdc):

        merging_condition = ' AND '.join([f'src.{i} = target.{i}' for i in key_cols])

        df_obj = DeltaTable.forName(spark,f'pyspark.silver.{table}')
        df_obj.alias('target').merge(
                df.alias('src'),
                merging_condition
            ).whenMatchedUpdateAll(condition = f'src.{cdc} >= target.{cdc}')\
            .whenNotMatchedInsertAll()\
            .execute()

        return 1   

#### **CUSTOMERS**

In [0]:
df = spark.read.table('pysparkdbt.bronze.customers')

In [0]:
display(df)

In [0]:
display(df.count())

In [0]:
df_cust = df.withColumn('domains',split(col("email"),'@')[1])

In [0]:
display(df_cust)

In [0]:
df_cust = df_cust.withColumn('phone_number',regexp_replace('phone_number',r'[^0-9]',''))

In [0]:
df_cust = df_cust.withColumn('full_name', concat_ws(' ', col('first_name'), col('last_name')))
df_cust.drop('first_name', 'last_name')

In [0]:
display(df_cust)

In [0]:
cust_obj = custom_transformations()

In [0]:
df_cust = cust_obj.dedup(df_cust, ['customer_id'], 'last_updated_timestamp')

In [0]:
display(df_cust)

In [0]:
df_cust = cust_obj.proceess_date(df_cust)

In [0]:
from delta.tables import DeltaTable

if not spark.catalog.tableExists('pysparkdbt.silver.customers'):

    df_cust.write.format('delta')\
        .mode('append')\
        .saveAsTable('pysparkdbt.silver.customers')
    
else:
    cust_obj.upsert(df_cust,['customer_id'],'customers','last_updated_timestamp')


In [0]:
%sql
select  count(*) from pysparkdbt.silver.customers

#### **Drivers**

In [0]:
df_drivers = spark.read.table('pysparkdbt.bronze.drivers')

In [0]:
display(df_drivers)

In [0]:
df_drivers = df_drivers.withColumn('full_name', concat_ws(' ', col('first_name'), col('last_name')))


In [0]:
df_drivers = df_drivers.drop('first_name', 'last_name')

In [0]:
df_drivers = df_drivers.withColumn('phone_number',regexp_replace('phone_number',r'[^0-9]',''))

In [0]:
df_drivers_obj = custom_transformations()

In [0]:
df_drivers = df_drivers_obj.dedup(df_drivers, ['driver_id'], 'last_updated_timestamp')

In [0]:
df_drivers = df_drivers_obj.proceess_date(df_drivers)

In [0]:
from delta.tables import DeltaTable

if not spark.catalog.tableExists('pysparkdbt.silver.drivers'):

    df_drivers.write.format('delta')\
        .mode('append')\
        .saveAsTable('pysparkdbt.silver.drivers')
    
else:
    df_drivers_obj.upsert(df_drivers,['driver_id'],'drivers','last_updated_timestamp')


In [0]:

display(df_drivers)

In [0]:
%sql
select count(*) from pysparkdbt.silver.drivers

#### **Location**

In [0]:
df_locations = spark.read.table('pysparkdbt.bronze.locations')

In [0]:
display(df_locations)

In [0]:
loc_obj = custom_transformations()

df_locations = loc_obj.proceess_date(df_locations)

In [0]:
if not spark.catalog.tableExists('pysparkdbt.silver.locations'):
    df_drivers.write.format('delta')\
        .mode('append')\
        .saveAsTable('pysparkdbt.silver.locations')
else:
    df_drivers_obj.upsert(df_locations,['location_id'],'locations','last_updated_timestamp')


In [0]:
%sql
select count(*) from pysparkdbt.silver.locations

#### **Payments**

In [0]:
df_payments = spark.read.table('pysparkdbt.bronze.payments')

In [0]:
display(df_payments)

In [0]:
from pyspark.sql.functions import col, when

df_payments = df_payments.withColumn(
    'online_payments',
    when((col('payment_method') == 'Card') & (col('payment_status') == 'Success'), 'online success')
    .when((col('payment_method') == 'Card') & (col('payment_status') == 'Failed'), 'online failed')
    .when((col('payment_method') == 'Card') & (col('payment_status') == 'Pending'), 'online pending')
    .otherwise('offline')
)

pay_obj = custom_transformations()
df_payments = pay_obj.proceess_date(df_payments)
df_payments = pay_obj.dedup(df_payments,['payment_id'],'last_updated_timestamp')

if not spark.catalog.tableExists('pysparkdbt.silver.payments'):
    df_payments.write.format('delta')\
        .mode('append')\
        .saveAsTable('pysparkdbt.silver.payments')
else:
    pay_obj.upsert(df_payments, ['payment_id'], 'payments', 'last_updated_timestamp')

In [0]:
%sql
select count(*)  from pysparkdbt.silver.payments

#### **Vehicles**

In [0]:
df_veh = spark.read.table('pysparkdbt.bronze.vehicles')

In [0]:
display(df_veh)

In [0]:
df_veh = df_veh.withColumn('make',upper(col('make')))

In [0]:
veh_obj = custom_transformations()
df_veh = veh_obj.proceess_date(df_veh)
df_veh = veh_obj.dedup(df_veh,['vehicle_id'],'last_updated_timestamp')

if not spark.catalog.tableExists('pysparkdbt.silver.vehicles'):
    df_veh.write.format('delta')\
        .mode('append')\
        .saveAsTable('pysparkdbt.silver.vehicles')
else:
    veh_obj.upsert(df_veh, ['vehicle_id'], 'vehicles', 'last_updated_timestamp')

In [0]:
%sql
select count(*) from pysparkdbt.silver.vehicles